# 01. 외삽 (Extrapolation)

**목표:** 외삽이 무엇인지, 보간과 무엇이 다른지, 왜 시간이 지날수록 틀리는지 직접 그려 보며 이해한다.

순서
1. 아는 점 몇 개로 시작하기
2. 보간: 두 점 **사이**를 채우기
3. 외삽: 마지막 점 **다음**을 예상하기
4. 외삽이 틀리는 이유: 가정이 깨질 때
5. 정리와 다음 단계(추측항법)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 글꼴 (macOS 기본 글꼴)
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. 아는 점 몇 개로 시작하기

배 한 척이 직선 위를 움직인다고 하자. 10초마다 위치 보고가 온다.

| 시각 (초) | 0 | 10 | 20 |
|---|---|---|---|
| 위치 (m) | 0 | 60 | 120 |

10초에 60m씩 가므로 속력은 **6 m/s**(약 11.7노트)다.

In [ ]:
t_known = np.array([0, 10, 20])      # 보고가 온 시각 (초)
x_known = np.array([0, 60, 120])     # 그때 위치 (m)

plt.plot(t_known, x_known, "o", markersize=9, label="받은 보고")
plt.xlabel("시각 (초)"); plt.ylabel("위치 (m)")
plt.xlim(-2, 42); plt.ylim(-10, 260)
plt.grid(alpha=.3); plt.legend(); plt.show()

## 2. 보간: 두 점 사이를 채우기

15초 위치는? 10초(60m)와 20초(120m) 사이에 있다. **두 점을 모두 알고 있으니** 그 사이를 직선으로 이으면 된다.

$$x(t) = x_1 + \frac{t - t_1}{t_2 - t_1}(x_2 - x_1)$$

15초는 10초와 20초의 딱 가운데이므로 60과 120의 가운데인 **90m**다.

In [ ]:
def interpolate(t, t1, x1, t2, x2):
    """두 점 (t1, x1), (t2, x2) 사이의 t 에서의 값 (선형 보간)."""
    return x1 + (t - t1) / (t2 - t1) * (x2 - x1)

interpolate(15, 10, 60, 20, 120)

## 3. 외삽: 마지막 점 다음을 예상하기

이제 30초 위치는? 마지막 보고는 20초다. **다음 보고가 아직 없으니** 사이를 채울 수 없다.

대신 이렇게 **가정**한다: *"지금까지처럼 6 m/s로 계속 간다."*

$$x(t) = x_{\text{마지막}} + v \times (t - t_{\text{마지막}})$$

30초면 120 + 6 × 10 = **180m**. 이것이 외삽이다. 핵심은 수식이 아니라 **가정**이다.

In [ ]:
def extrapolate(t, t_last, x_last, v):
    """마지막 점 (t_last, x_last)에서 속력 v 로 계속 갔다고 보고 t 에서의 값 (선형 외삽)."""
    return x_last + v * (t - t_last)

v = (x_known[-1] - x_known[-2]) / (t_known[-1] - t_known[-2])   # 마지막 두 보고로 구한 속력
print("속력:", v, "m/s")
print("30초 예상 위치:", extrapolate(30, 20, 120, v), "m")

In [ ]:
t_inter = np.linspace(0, 20, 50)
t_extra = np.linspace(20, 40, 50)

plt.plot(t_inter, np.interp(t_inter, t_known, x_known), "-", label="보간 (사이를 채움)")
plt.plot(t_extra, extrapolate(t_extra, 20, 120, v), "--", label="외삽 (앞을 예상)")
plt.plot(t_known, x_known, "o", markersize=9, color="k", label="받은 보고")
plt.axvline(20, color="gray", lw=1); plt.text(20.5, 10, "마지막 보고", color="gray")
plt.xlabel("시각 (초)"); plt.ylabel("위치 (m)")
plt.xlim(-2, 42); plt.ylim(-10, 260)
plt.grid(alpha=.3); plt.legend(); plt.show()

| | 보간 | 외삽 |
|---|---|---|
| 아는 것 | 앞뒤 두 점 | 마지막 점(과 그 전까지의 흐름) |
| 구하는 곳 | 두 점 **사이** | 마지막 점 **다음** |
| 틀리는 정도 | 두 점이 붙잡아 줘서 작다 | 시간이 갈수록 커질 수 있다 |
| 배 위치에서 | 지난 항적을 매끄럽게 그릴 때 | 다음 보고 전까지 **지금** 위치를 그릴 때 |

## 4. 외삽이 틀리는 이유: 가정이 깨질 때

실제 배가 20초부터 **속력을 줄였다**고 하자(6 m/s → 점점 감속). 외삽은 이 사실을 모른다. 다음 보고가 오기 전까지는 "6 m/s로 계속 간다"는 가정을 그대로 쓴다.

In [ ]:
def true_position(t):
    """실제 위치: 20초까지 6 m/s, 그 뒤로 매초 0.2 m/s 씩 감속."""
    t = np.asarray(t, dtype=float)
    after = np.clip(t - 20, 0, None)
    return np.where(t <= 20, 6 * t, 120 + 6 * after - 0.1 * after**2)

t = np.linspace(0, 40, 200)
plt.plot(t, true_position(t), "-", lw=2, color="gray", label="실제 위치")
plt.plot(t_extra, extrapolate(t_extra, 20, 120, v), "--", label="외삽")
plt.plot(t_known, x_known, "o", markersize=9, color="k", label="받은 보고")
plt.xlabel("시각 (초)"); plt.ylabel("위치 (m)")
plt.grid(alpha=.3); plt.legend(); plt.show()

for s in [22, 25, 30, 40]:
    err = extrapolate(s, 20, 120, v) - true_position(s)
    print(f"{s}초: 오차 {err:5.1f} m")

오차가 **시간의 제곱**으로 커진다(2초 0.4m → 20초 40m). 외삽은 마지막 보고에서 멀어질수록 틀린다.

그래서 실제로는
- 새 보고가 오면 바로 그 위치로 **다시 시작**한다.
- 너무 오래 보고가 없으면 **외삽을 멈춘다**.
- 가정을 더 똑똑하게 만든다(속력만이 아니라 도는 정도까지) → 곡선 추측항법, 칼만 필터.

## 직접 해 보기

1. `true_position`의 감속값 `0.1`을 `0`으로 바꾸면 오차는 어떻게 되나? 왜 그런가?
2. 감속 대신 **가속**(`-0.1` → `+0.1`)이면 외삽은 실제보다 앞서나, 뒤처지나?
3. 보고 간격이 10초가 아니라 **60초**라면, 다음 보고 직전 오차는 몇 m인가?

In [ ]:
# 여기서 직접 바꿔 보세요


## 정리

- **보간**: 아는 두 점 사이를 채운다.
- **외삽**: 마지막으로 아는 점 다음을 **가정**으로 예상한다.
- 외삽의 정확도는 가정이 얼마나 맞느냐에 달려 있고, 시간이 지날수록 틀어진다.

**다음 (02):** 이 1차원 외삽을 바다 위 2차원으로 넓힌다. 속력에 **방향(COG)**이 붙으면 그것이 **추측항법(직선 DR)**이다.